# Desafiá al motor de evaluación de impacto (CCAM)

Este notebook levanta el motor de la Pre-Entrega 2 en Google Colab para que puedas probarlo con tus propios pedidos de cambio.

**Antes de empezar:** abrí el ícono de la llave 🔑 en la barra lateral, creá el secret `ANTHROPIC_API_KEY` con tu clave de Anthropic y habilitale el acceso a este notebook. Sin clave igual podés usar el modo simulado (celda 5).

Todos los datos son ficticios.

## 1. Descargar el repositorio e instalar

In [ ]:
REPO = "https://github.com/CFR21-VSC/ccam-llm.git"
!git clone -q --branch pe2 $REPO fde-control-cambios || git clone -q $REPO fde-control-cambios
%cd fde-control-cambios
!pip -q install -r requirements.txt

## 2. Cargar la clave desde Secrets (nunca se imprime)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["LLM_PROVIDER"] = "anthropic"
    print("Clave cargada desde Secrets.")
except Exception:
    os.environ["LLM_PROVIDER"] = "simulado"
    print("Sin clave: se usa el modo simulado.")
os.environ.setdefault("LLM_MODEL", "claude-haiku-5-5")

## 3. Las pruebas automáticas (no gastan tokens)

In [ ]:
!python -m pytest -q

## 4. Tu propio CCAM

Editá los campos y corré la celda. Probá romperlo: una descripción corta, un tipo de cambio que no existe, un pedido que no sea un cambio, o pedile que cite un SOP inventado.

In [ ]:
from src.motor import evaluar

mi_ccam = {
    "id_ccam": "CCAM-2026-301",
    "tipo_cambio": "equipo_instalacion",        # version_software, configuracion_sistema, packaging_rotulado, proveedor_material, equipo_instalacion, documento_procedimiento, proceso_productivo, otro
    "area_solicitante": "produccion",            # produccion, control_calidad, garantia_calidad, validaciones, sistemas, mantenimiento, deposito, compras, asuntos_regulatorios
    "sistema_afectado": "Comprimidora rotativa de la línea 2",
    "descripcion": "Reemplazar los punzones de la comprimidora rotativa de la línea 2 por un juego de otro fabricante con el mismo diseño de cara.",
}

r = evaluar(mi_ccam)
print(r.model_dump_json(indent=2, exclude_none=True))

## 5. Ver los caminos de error sin clave

In [ ]:
from src.proveedores import ProveedorSimulado
for escenario in ["invalido_luego_valido", "invalido", "sin_conexion", "limite_uso"]:
    r = evaluar(mi_ccam, ProveedorSimulado(escenario))
    print(f"{escenario:<22} ok={r.ok}  error={r.error.error if r.error else '-'}  intentos={r.metricas.intentos if r.metricas else 0}")

## 6. La corrida de medición (15 casos)

In [ ]:
!python -m scripts.corrida